In [ ]:
import matplotlib as mtp
import pandas as pd
import numpy as np

opay = pd.read_csv("../archive/olist_order_payments_dataset.csv") #
ordit = pd.read_csv("../archive/olist_order_items_dataset.csv") #
orev = pd.read_csv("../archive/olist_order_reviews_dataset.csv") #
osel = pd.read_csv("../archive/olist_sellers_dataset.csv") # 
oprod =  pd.read_csv("../archive/olist_products_dataset.csv") #
orders =  pd.read_csv("../archive/olist_orders_dataset.csv") #

opay.drop_duplicates()
ordit.drop_duplicates()
orev.drop_duplicates()
osel.drop_duplicates()
oprod.drop_duplicates()
orders.drop_duplicates()

opay = opay.replace(r'^\s*$', np.nan, regex=True)
ordit = ordit.replace(r'^\s*$', np.nan, regex=True)
orev = orev.replace(r'^\s*$', np.nan, regex=True)
osel = osel.replace(r'^\s*$', np.nan, regex=True)
oprod = oprod.replace(r'^\s*$', np.nan, regex=True)
orders = orders.replace(r'^\s*$', np.nan, regex=True)

orev["review_comment_message"] = orev["review_comment_message"].replace(
    r"^\s*$", np.nan, regex=True
)
assert not orev["review_comment_message"].dropna().str.fullmatch(r"\s*").any()

In [ ]:
# número de reviews com nota menor que 4
menor_q_4 = orev.loc[orev['review_score'] < 4]
print(f"{len(menor_q_4)} reviews com nota menor que 4")

distribuicao =(
    menor_q_4.groupby('review_score')
    .agg(
        qtd=('review_score','count')
    )
)
# porcentagem de reviews com nota menor que 4
media_menor_que_4 = len(orev.query("review_score <  4")) / len(orev) * 100
print(f"{media_menor_que_4:.1f}% menor que 4")

# média geral das reviews
media_geral = float(orev["review_score"].mean())
print(f"{media_geral:.2f}/5 - média geral das reviews")
# truncado = float(str(media)[: str(media).find(".") + 3])

condicao = (
    (orev["review_score"] < 4)
    & orev["review_comment_message"].notna()
    & orev["review_comment_message"].str.strip().ne("")
)

# below4comments = condicao.mean()
# below4comments

complaints = pd.DataFrame(orev.loc[condicao])
complaints = complaints.drop_duplicates()
complaints
distribuicao


Gerando um novo csv com os dados limpos que precisamos da tabela de reviews

In [ ]:
df = (
    complaints[['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']]
    .merge(
        ordit[['order_id','order_item_id','product_id','seller_id','price','freight_value']],
        on='order_id',
        how='left'
    ).merge(
        osel[['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']],
        on='seller_id',
        how='left'
    ).merge(
        oprod[['product_id','product_category_name']],
        on='product_id',
        how='left'
    ).merge(
        opay,
        on='order_id',
        how='left'
    ).merge(
        orders[['order_id','order_status','order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date']],
        on='order_id',
        how='left',
    )
)

df = df.drop_duplicates()
df['seller_zip_code_prefix'] = df['seller_zip_code_prefix'].astype(str).str.strip()  
df['order_item_id'] = df['order_item_id'].astype('Int64')
df['payment_sequential'] = df['payment_sequential'].astype('Int64')
df['payment_installments'] = df['payment_installments'].astype('Int64')
df.dropna(how="all")

df.shape

Gerando uma nova tabela unindo dados de outras para uma melhor análise

In [ ]:
# !!! PROBLEMA: Estou encontrando muitas reviews e orders repetidas.

# pd.unique(df['review_id'])
rep_an = df[
    [
        'review_id',
        'order_id',
        'price',
        'freight_value',
        'payment_value',
        'payment_type',
        'review_comment_message',
        'review_score',
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
].dropna(subset='price')
rep_an

Criando as tabelas

In [ ]:

complaints.to_csv("created-tables/complained_reviews.csv")
df.to_csv("created-tables/complained_reviews_full_data.csv")
rep_an.to_csv('created-tables/repeated_values_analysis.csv')